# Azure AI Content Safety: Detect Harmful Text and Images

In this notebook you will learn how to use **Azure AI Content Safety** to:

1. Analyze **text** for harmful content (hate, violence, self-harm, sexual)
2. Analyze **images** for harmful content
3. Interpret the severity scores returned by the service

---

### Prerequisites: Deploy the Azure Resource

Before running this notebook, you need an **Azure AI Content Safety** resource. Instead of configuring it manually in the Azure Portal, deploy the included **ARM template** which creates the resource in the correct configuration automatically.

> **Tip:** Open `deploy-content-safety.parameters.json` to customize the resource name, region, or SKU before deploying. The default SKU is **S0** (paid). Change it to **F0** for the free tier.

**1. Deploy the resource** — run the cell below (replace `<your-resource-group>`):

In [1]:
import subprocess, shutil

# ⚠️ Replace <your-resource-group> with your actual resource group name
RESOURCE_GROUP = "Foundry"

# Resolve the full path to the Azure CLI (needed for Windows where az is a .cmd file)
AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

result = subprocess.run(
    [AZ, "deployment", "group", "create",
     "--resource-group", RESOURCE_GROUP,
     "--template-file", "deploy-content-safety.json",
     "--parameters", "deploy-content-safety.parameters.json"],
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

{
  "id": "/subscriptions/ff86f0f3-cbf4-4641-9845-300dd46ca8c9/resourceGroups/Foundry/providers/Microsoft.Resources/deployments/deploy-content-safety",
  "location": null,
  "name": "deploy-content-safety",
  "properties": {
    "correlationId": "5233df06-e73e-49b8-9a45-be8de41f7e6d",
    "debugSetting": null,
    "dependencies": [],
    "diagnostics": null,
    "duration": "PT1.3787445S",
    "error": null,
    "extensions": [],
    "mode": "Incremental",
    "onErrorDeployment": null,
    "outputResources": [
      {
        "apiVersion": null,
        "extension": null,
        "id": "/subscriptions/ff86f0f3-cbf4-4641-9845-300dd46ca8c9/resourceGroups/Foundry/providers/Microsoft.CognitiveServices/accounts/foundry-demo-content-safety",
        "identifiers": null,
        "resourceGroup": "Foundry",
        "resourceType": "Microsoft.CognitiveServices/accounts"
      }
    ],
    "outputs": {
      "contentSafetyEndpoint": {
        "type": "String",
        "value": "https://foundry-

**2. Assign the required RBAC role** so your identity can use the resource (replace `<your-resource-group>`):

> **Note:** RBAC role assignments can take **1–2 minutes** to propagate. If you get a 401 error on your first run, wait a moment and try again.

In [2]:
import subprocess, shutil

RESOURCE_GROUP = "Foundry"
AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

# Step A: Get the signed-in user's Object ID
user_result = subprocess.run(
    [AZ, "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"],
    capture_output=True, text=True, encoding="utf-8"
)
user_id = user_result.stdout.strip()
print(f"Your User ID: {user_id}")

# Step B: Get the resource ID from the deployment output
scope_result = subprocess.run(
    [AZ, "deployment", "group", "show",
     "--resource-group", RESOURCE_GROUP,
     "--name", "deploy-content-safety",
     "--query", "properties.outputs.resourceId.value", "-o", "tsv"],
    capture_output=True, text=True, encoding="utf-8"
)
resource_id = scope_result.stdout.strip()
print(f"Resource ID:  {resource_id}")

# Step C: Assign the RBAC role
role_result = subprocess.run(
    [AZ, "role", "assignment", "create",
     "--assignee", user_id,
     "--role", "Cognitive Services User",
     "--scope", resource_id],
    capture_output=True, text=True, encoding="utf-8"
)
print(role_result.stdout)
if role_result.returncode != 0:
    print("ERROR:", role_result.stderr)

Your User ID: c417f445-fdf8-46d3-8bd0-4cbf2dd485b1
Resource ID:  /subscriptions/ff86f0f3-cbf4-4641-9845-300dd46ca8c9/resourceGroups/Foundry/providers/Microsoft.CognitiveServices/accounts/foundry-demo-content-safety
{
  "condition": null,
  "conditionVersion": null,
  "createdBy": "c417f445-fdf8-46d3-8bd0-4cbf2dd485b1",
  "createdOn": "2026-02-18T15:26:35.383591+00:00",
  "delegatedManagedIdentityResourceId": null,
  "description": null,
  "id": "/subscriptions/ff86f0f3-cbf4-4641-9845-300dd46ca8c9/resourceGroups/Foundry/providers/Microsoft.CognitiveServices/accounts/foundry-demo-content-safety/providers/Microsoft.Authorization/roleAssignments/601a119a-9518-4177-970a-13898258d1b0",
  "name": "601a119a-9518-4177-970a-13898258d1b0",
  "principalId": "c417f445-fdf8-46d3-8bd0-4cbf2dd485b1",
  "principalName": "Chris@azuredemosolutions.com",
  "principalType": "User",
  "resourceGroup": "Foundry",
  "roleDefinitionId": "/subscriptions/ff86f0f3-cbf4-4641-9845-300dd46ca8c9/providers/Microsoft.A

**3. Update the `.env` file** in this folder with the endpoint from the deployment output:

```
CONTENT_SAFETY_ENDPOINT="https://<your-account-name>.cognitiveservices.azure.com/"
```

You can retrieve this value by running the cell below (replace `<your-resource-group>`):

In [3]:
import subprocess, shutil

RESOURCE_GROUP = "Foundry"
AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

result = subprocess.run(
    [AZ, "deployment", "group", "show",
     "--resource-group", RESOURCE_GROUP,
     "--name", "deploy-content-safety",
     "--query", "properties.outputs"],
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

{
  "contentSafetyEndpoint": {
    "type": "String",
    "value": "https://foundry-demo-content-safety.cognitiveservices.azure.com/"
  },
  "resourceId": {
    "type": "String",
    "value": "/subscriptions/ff86f0f3-cbf4-4641-9845-300dd46ca8c9/resourceGroups/Foundry/providers/Microsoft.CognitiveServices/accounts/foundry-demo-content-safety"
  }
}



### Authentication

This notebook uses **Microsoft Entra ID** (`DefaultAzureCredential`) instead of API keys.
Make sure you are logged in with `az login`.

### How Severity Scores Work

The Content Safety API returns a severity score from **0 to 6** for each
category. The higher the score, the more harmful the content:

| Score | Meaning |
|-------|---------|
| 0     | Safe    |
| 2     | Low     |
| 4     | Medium  |
| 6     | High    |

## Step 1: Install the Content Safety SDK

In [4]:
%pip install azure-ai-contentsafety azure-identity python-dotenv

  Using cached azure_ai_contentsafety-1.0.0-py3-none-any.whl.metadata (30 kB)
Using cached azure_ai_contentsafety-1.0.0-py3-none-any.whl (61 kB)

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 2: Load Environment Variables and Create the Client

In [5]:
import os
from dotenv import load_dotenv
from azure.ai.contentsafety import ContentSafetyClient
from azure.identity import DefaultAzureCredential

load_dotenv(override=True)

safety_endpoint = os.getenv("CONTENT_SAFETY_ENDPOINT")

# Authenticate via Entra ID (uses az login credentials)
credential = DefaultAzureCredential()

safety_client = ContentSafetyClient(
    safety_endpoint, credential
)

print("Endpoint:", safety_endpoint)
print("Client ready:", safety_client is not None)

Endpoint: https://foundry-demo-content-safety.cognitiveservices.azure.com/
Client ready: True


## Step 3: Analyze Text for Harmful Content

We send several sample sentences to the Content Safety API and examine
the severity scores for each category.

In [6]:
from azure.ai.contentsafety.models import AnalyzeTextOptions

# A helper function to display results in a readable table
def display_text_analysis(label, result):
    """Print a formatted table of category severity scores."""
    print(f"\n--- {label} ---")
    print(f"  {'Category':<15} {'Severity':>8}")
    print(f"  {'-'*25}")
    for item in result.categories_analysis:
        print(f"  {item.category:<15} {item.severity:>8}")

In [ ]:
# Example 1: A harmless sentence (expect all zeros)
safe_text = "The weather is lovely today and I enjoyed my morning coffee."
safe_result = safety_client.analyze_text(
    AnalyzeTextOptions(text=safe_text)
)
display_text_analysis("Safe Text", safe_result)

# Example 2: A potentially concerning sentence
concerning_text = "I am feeling very isolated."
concerning_result = safety_client.analyze_text(
    AnalyzeTextOptions(text=concerning_text)
)
display_text_analysis("Concerning Text", concerning_result)

# Example 3: A hateful sentence
hateful_text = "People from that group are all terrible and should be fired."
hateful_result = safety_client.analyze_text(
    AnalyzeTextOptions(text=hateful_text)
)
display_text_analysis("Hateful Text", hateful_result)


--- Safe Text ---
  Category        Severity
  -------------------------
  Hate                   0
  SelfHarm               0
  Sexual                 0
  Violence               0

--- Concerning Text ---
  Category        Severity
  -------------------------
  Hate                   0
  SelfHarm               0
  Sexual                 0
  Violence               0

--- Hateful Text ---
  Category        Severity
  -------------------------
  Hate                   4
  SelfHarm               0
  Sexual                 0
  Violence               0


## Step 4: Build a Simple Content Filter

In production you would typically set a **threshold** and reject or flag
content that exceeds it. Here is a simple helper function that does this.

In [ ]:
def is_content_safe(text: str, threshold: int = 2) -> bool:
    """
    Return True if every category severity is below the threshold.

    Parameters
    ----------
    text : str
        The text to analyze.
    threshold : int
        Maximum allowed severity (default 2 = low).
    """
    result = safety_client.analyze_text(AnalyzeTextOptions(text=text))
    for item in result.categories_analysis:
        if item.severity >= threshold:
            return False
    return True


# Test our filter
test_messages = [
    "Let's schedule a team meeting for Monday morning.",
    "I want to physically hurt the person who cut me off in traffic.",
    "Can you help me write a thank-you note to my teacher?",
]

for msg in test_messages:
    safe = is_content_safe(msg)
    status = "SAFE" if safe else "BLOCKED"
    print(f"[{status:>7}]  {msg}")

## Step 5: Analyze an Image for Harmful Content

The Content Safety API can also scan images. We read the `foundry.png`
image file included in this folder and send its bytes to the service.

In [ ]:
from azure.ai.contentsafety.models import AnalyzeImageOptions, ImageData

# Read a local image file
image_path = "foundry.png"

if os.path.exists(image_path):
    with open(image_path, "rb") as img_file:
        image_bytes = img_file.read()

    image_request = AnalyzeImageOptions(
        image=ImageData(content=image_bytes)
    )
    image_result = safety_client.analyze_image(image_request)

    print(f"Image analysis for '{image_path}':")
    print(f"  {'Category':<15} {'Severity':>8}")
    print(f"  {'-'*25}")
    for item in image_result.categories_analysis:
        print(f"  {item.category:<15} {item.severity:>8}")
else:
    print(f"Image file '{image_path}' not found.")
    print("Place a .png image in this folder and re-run, or update the path above.")